# Power Automate Control Flow Simulator

## Learning Objectives

By completing this notebook, you will:

1. Simulate Power Automate branching, loop, and error handling patterns in Python
2. Understand how Condition, Switch, Apply to Each, and Do Until translate to code equivalents
3. Build a minimal flow executor that tracks execution paths and action states
4. Visualize flow execution graphs with matplotlib
5. Practice diagnosing common control flow errors by reading simulated run history

## Prerequisites

- Module 04 guides completed (branching, loops, error handling)
- Python basics: functions, dictionaries, lists, exceptions
- No Power Automate account required — everything runs locally

## Estimated Time: 15 minutes

---

In [ ]:
learning_objectives([
    "Module 04 guides completed (branching, loops, error handling)",
    "Python basics: functions, dictionaries, lists, exceptions",
    "No Power Automate account required — everything runs locally"
])

## 1. Setup

We use only standard library modules plus matplotlib for visualization. No external API calls needed.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Standard library only — no pip installs needed
import random
import time
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Callable, Dict, List, Optional

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

print("Setup complete.")

## 2. Core Data Model

Power Automate tracks every action's outcome as one of four states. We model that here.

**Power Automate state → Python equivalent:**

| Power Automate | Our Simulator |
|---|---|
| Succeeded | `ActionStatus.SUCCEEDED` |
| Failed | `ActionStatus.FAILED` |
| Skipped | `ActionStatus.SKIPPED` |
| Timed Out | `ActionStatus.TIMED_OUT` |

In [ ]:
# ActionStatus mirrors Power Automate's four possible action outcomes.
# Every action in a flow run lands in exactly one of these states.

class ActionStatus(Enum):
    SUCCEEDED = "Succeeded"
    FAILED = "Failed"
    SKIPPED = "Skipped"
    TIMED_OUT = "Timed Out"


@dataclass
class ActionResult:
    """
    Result of a single action execution, mirroring what Power Automate
    shows in the run history detail panel for each action.
    """
    name: str
    status: ActionStatus
    output: Any = None
    error: Optional[str] = None
    duration_ms: int = 0

    def __repr__(self) -> str:
        parts = [f"{self.name}: {self.status.value}"]
        if self.error:
            parts.append(f"  error={self.error!r}")
        if self.output is not None:
            parts.append(f"  output={self.output!r}")
        return "\n".join(parts)


@dataclass
class FlowRun:
    """
    Represents a single flow execution. Collects action results in order,
    mirroring the run history timeline in Power Automate.
    """
    trigger_name: str
    trigger_data: Dict[str, Any]
    action_results: List[ActionResult] = field(default_factory=list)

    def record(self, result: ActionResult) -> None:
        """Append an action result to the run history."""
        self.action_results.append(result)

    def last_status(self) -> Optional[ActionStatus]:
        """Return the status of the most recently completed action."""
        if not self.action_results:
            return None
        return self.action_results[-1].status

    def print_history(self) -> None:
        """Print run history like Power Automate's run detail view."""
        print(f"Trigger: {self.trigger_name}")
        print(f"Data: {self.trigger_data}")
        print("-" * 40)
        for result in self.action_results:
            status_icon = {"Succeeded": "✓", "Failed": "✗", "Skipped": "⊘", "Timed Out": "⏱"}
            icon = status_icon.get(result.status.value, "?")
            print(f"  {icon} {result.name}: {result.status.value}", end="")
            if result.error:
                print(f" — {result.error}", end="")
            print()


print("Data model defined.")

## 3. The Flow Executor

This executor implements the core control flow primitives as Python methods.

Each method maps directly to a Power Automate action type:

| `FlowExecutor` method | Power Automate equivalent |
|---|---|
| `run_action()` | Any single action |
| `condition()` | Condition action |
| `switch()` | Switch action |
| `apply_to_each()` | Apply to Each |
| `do_until()` | Do Until |
| `scope()` | Scope action |
| `run_after()` | Configure Run After |
| `terminate()` | Terminate action |

In [ ]:
# FlowExecutor: the core simulator.
#
# Design principle: each method signature mirrors how you configure the
# equivalent action in Power Automate. run_action wraps an arbitrary
# callable, adding status tracking and run-after logic automatically.

class FlowExecutor:
    """
    Simulates Power Automate flow execution logic.

    Methods correspond to Power Automate control flow actions.
    All actions record their outcomes to a FlowRun for later inspection.
    """

    def __init__(self, run: FlowRun) -> None:
        self.run = run
        # Stores the last action result for run_after checks
        self._last_result: Optional[ActionResult] = None

    # ------------------------------------------------------------------
    # Primitive: run a single action
    # ------------------------------------------------------------------

    def run_action(
        self,
        name: str,
        fn: Callable[[], Any],
        run_after: Optional[List[ActionStatus]] = None,
        timeout_ms: int = 5000,
    ) -> ActionResult:
        """
        Execute a single action.

        Parameters
        ----------
        name : str
            Action display name (shown in run history).
        fn : callable
            The function that implements the action's logic.
        run_after : list of ActionStatus, optional
            States of the previous action that allow this action to run.
            Default: [SUCCEEDED] — mirrors Power Automate's default.
        timeout_ms : int
            Simulated timeout in milliseconds.

        Returns
        -------
        ActionResult
            The outcome of this action.
        """
        if run_after is None:
            run_after = [ActionStatus.SUCCEEDED]

        # Apply run-after logic: skip if predecessor state is not allowed
        if self._last_result is not None:
            if self._last_result.status not in run_after:
                result = ActionResult(
                    name=name,
                    status=ActionStatus.SKIPPED,
                )
                self.run.record(result)
                self._last_result = result
                return result

        # Execute the action
        start = time.monotonic()
        try:
            output = fn()
            elapsed_ms = int((time.monotonic() - start) * 1000)
            if elapsed_ms > timeout_ms:
                result = ActionResult(
                    name=name,
                    status=ActionStatus.TIMED_OUT,
                    duration_ms=elapsed_ms,
                    error=f"Action exceeded {timeout_ms}ms timeout",
                )
            else:
                result = ActionResult(
                    name=name,
                    status=ActionStatus.SUCCEEDED,
                    output=output,
                    duration_ms=elapsed_ms,
                )
        except Exception as exc:
            elapsed_ms = int((time.monotonic() - start) * 1000)
            result = ActionResult(
                name=name,
                status=ActionStatus.FAILED,
                error=str(exc),
                duration_ms=elapsed_ms,
            )

        self.run.record(result)
        self._last_result = result
        return result

    # ------------------------------------------------------------------
    # Condition: binary branching
    # ------------------------------------------------------------------

    def condition(
        self,
        name: str,
        expression: bool,
        yes_branch: Callable[[], None],
        no_branch: Optional[Callable[[], None]] = None,
    ) -> ActionResult:
        """
        Binary Yes/No branching — equivalent to Power Automate Condition action.

        Parameters
        ----------
        name : str
            Condition name for run history.
        expression : bool
            Result of evaluating the condition (True = Yes branch).
        yes_branch : callable
            Actions to run when expression is True.
        no_branch : callable, optional
            Actions to run when expression is False.
        """
        result = ActionResult(
            name=name,
            status=ActionStatus.SUCCEEDED,
            output={"branch": "Yes" if expression else "No"},
        )
        self.run.record(result)
        self._last_result = result

        if expression:
            yes_branch()
        elif no_branch is not None:
            no_branch()

        return result

    # ------------------------------------------------------------------
    # Switch: multi-path routing
    # ------------------------------------------------------------------

    def switch(
        self,
        name: str,
        value: Any,
        cases: Dict[Any, Callable[[], None]],
        default: Optional[Callable[[], None]] = None,
    ) -> ActionResult:
        """
        Multi-path routing — equivalent to Power Automate Switch action.

        Parameters
        ----------
        name : str
            Switch name for run history.
        value : any
            The value being switched on (equivalent to the 'On' field).
        cases : dict
            Mapping of case values to callables — each key is a Case Equals value.
        default : callable, optional
            Runs when no case matches — equivalent to the Default case.
        """
        matched_case = cases.get(value)
        result = ActionResult(
            name=name,
            status=ActionStatus.SUCCEEDED,
            output={"on": value, "matched_case": value if matched_case else "Default"},
        )
        self.run.record(result)
        self._last_result = result

        if matched_case:
            matched_case()
        elif default:
            default()

        return result

    # ------------------------------------------------------------------
    # Apply to Each: iterate over an array
    # ------------------------------------------------------------------

    def apply_to_each(
        self,
        name: str,
        items: List[Any],
        body: Callable[[Any, int], None],
    ) -> ActionResult:
        """
        Iterate over an array — equivalent to Power Automate Apply to Each.

        Parameters
        ----------
        name : str
            Loop name for run history.
        items : list
            The array to iterate over.
        body : callable(item, index)
            Function called once per item. Receives the current item and
            its index — equivalent to using 'Current item' tokens.
        """
        loop_result = ActionResult(
            name=name,
            status=ActionStatus.SUCCEEDED,
            output={"item_count": len(items)},
        )

        for index, item in enumerate(items):
            try:
                body(item, index)
            except Exception as exc:
                # One item failing does not stop the whole loop by default
                # (mirrors Power Automate's behavior with default concurrency)
                iteration_result = ActionResult(
                    name=f"{name} — item {index + 1}",
                    status=ActionStatus.FAILED,
                    error=str(exc),
                )
                self.run.record(iteration_result)

        self.run.record(loop_result)
        self._last_result = loop_result
        return loop_result

    # ------------------------------------------------------------------
    # Do Until: loop with exit condition
    # ------------------------------------------------------------------

    def do_until(
        self,
        name: str,
        condition_fn: Callable[[], bool],
        body: Callable[[int], None],
        max_count: int = 60,
    ) -> ActionResult:
        """
        Loop until condition is true — equivalent to Power Automate Do Until.

        Parameters
        ----------
        name : str
            Loop name for run history.
        condition_fn : callable returning bool
            Exit condition. When this returns True, the loop stops.
        body : callable(iteration_number)
            Actions to run each iteration.
        max_count : int
            Safety limit — equivalent to Do Until's Count limit.
        """
        iteration = 0
        exit_reason = "condition_met"

        while not condition_fn():
            iteration += 1
            if iteration > max_count:
                exit_reason = "count_limit"
                break
            body(iteration)

        result = ActionResult(
            name=name,
            status=ActionStatus.SUCCEEDED,
            output={"iterations": iteration, "exit_reason": exit_reason},
        )
        self.run.record(result)
        self._last_result = result
        return result

    # ------------------------------------------------------------------
    # Scope: Try/Catch/Finally container
    # ------------------------------------------------------------------

    def scope(
        self,
        name: str,
        body: Callable[[], None],
    ) -> ActionResult:
        """
        Grouped action container — equivalent to Power Automate Scope.

        The scope succeeds if all contained actions succeed.
        It fails if any contained action fails (and that failure is not
        individually handled).
        """
        try:
            body()
            result = ActionResult(
                name=name,
                status=ActionStatus.SUCCEEDED,
            )
        except Exception as exc:
            result = ActionResult(
                name=name,
                status=ActionStatus.FAILED,
                error=str(exc),
            )

        self.run.record(result)
        self._last_result = result
        return result

    # ------------------------------------------------------------------
    # Terminate: hard stop
    # ------------------------------------------------------------------

    def terminate(self, status: str, message: str = "") -> None:
        """
        Stop the flow immediately — equivalent to Power Automate Terminate.

        Parameters
        ----------
        status : str
            One of 'Succeeded', 'Failed', or 'Cancelled'.
        message : str
            Human-readable reason for termination.
        """
        result = ActionResult(
            name="Terminate",
            status=ActionStatus.SUCCEEDED,
            output={"terminate_status": status, "message": message},
        )
        self.run.record(result)
        # Raise to stop execution — equivalent to flow run ending immediately
        raise StopIteration(f"Terminate({status}): {message}")


print("FlowExecutor defined.")

## 4. Pattern 1 — Condition (If/Else Branch)

This example simulates a flow that routes an invoice for approval based on its amount.

**Power Automate equivalent:**
- Trigger: When a new invoice is submitted
- Condition: `Amount` is greater than `5000`
  - Yes: Send email to CFO
  - No: Auto-approve and update status

In [ ]:
# Simulate two invoice submissions — one above threshold, one below.

def simulate_invoice_routing(amount: float) -> FlowRun:
    """Simulate the invoice routing flow for a given amount."""
    run = FlowRun(
        trigger_name="Invoice submitted",
        trigger_data={"invoice_id": "INV-2024-001", "amount": amount},
    )
    executor = FlowExecutor(run)

    # Action 1: Get invoice details
    executor.run_action(
        "Get invoice details",
        fn=lambda: {"id": "INV-2024-001", "amount": amount, "vendor": "Contoso Ltd"},
    )

    # Condition: amount > 5000?
    executor.condition(
        name="Amount > $5,000?",
        expression=amount > 5000,
        yes_branch=lambda: executor.run_action(
            "Send approval request to CFO",
            fn=lambda: {"email_sent_to": "cfo@contoso.com", "subject": f"Approval needed: ${amount:,.2f}"},
        ),
        no_branch=lambda: executor.run_action(
            "Auto-approve and update status",
            fn=lambda: {"status": "Approved", "approved_by": "Auto", "amount": amount},
        ),
    )

    # Action after condition — always runs
    executor.run_action(
        "Update invoice log",
        fn=lambda: {"logged": True, "amount": amount},
    )

    return run


print("=== Invoice: $3,200 (below threshold) ===")
run_low = simulate_invoice_routing(3200)
run_low.print_history()

print()
print("=== Invoice: $12,500 (above threshold) ===")
run_high = simulate_invoice_routing(12500)
run_high.print_history()

## 5. Pattern 2 — Switch (Multi-Path Routing)

Simulates the IT ticket routing flow from the guide.

**Power Automate equivalent:**
- Switch on `Category`
- Cases: Hardware, Software, Network, Security
- Default: General queue

In [ ]:
def simulate_ticket_routing(category: str, title: str) -> FlowRun:
    """Simulate IT ticket routing via Switch."""
    run = FlowRun(
        trigger_name="New IT ticket submitted",
        trigger_data={"title": title, "category": category},
    )
    executor = FlowExecutor(run)

    # Get ticket details
    executor.run_action(
        "Get ticket details",
        fn=lambda: {"title": title, "category": category, "id": "TKT-8821"},
    )

    # Switch on category
    executor.switch(
        name="Route by Category",
        value=category,
        cases={
            "Hardware": lambda: executor.run_action(
                "Post to #it-hardware",
                fn=lambda: {"channel": "#it-hardware", "message": title},
            ),
            "Software": lambda: executor.run_action(
                "Post to #it-software",
                fn=lambda: {"channel": "#it-software", "message": title},
            ),
            "Network": lambda: executor.run_action(
                "Post to #it-network",
                fn=lambda: {"channel": "#it-network", "message": title},
            ),
            "Security": lambda: [
                executor.run_action(
                    "Post to #it-security",
                    fn=lambda: {"channel": "#it-security", "message": title},
                ),
                executor.run_action(
                    "Page on-call security engineer",
                    fn=lambda: {"paged": True, "engineer": "on-call"},
                ),
            ],
        },
        default=lambda: executor.run_action(
            "Post to #it-general",
            fn=lambda: {"channel": "#it-general", "message": title},
        ),
    )

    executor.run_action(
        "Acknowledge ticket to submitter",
        fn=lambda: {"email_sent": True, "ticket_id": "TKT-8821"},
    )

    return run


tickets = [
    ("Hardware", "Laptop screen flickering"),
    ("Security", "Suspicious login attempt detected"),
    ("Printing", "Printer not connecting"),   # Will hit Default
]

for category, title in tickets:
    print(f"\n=== Ticket: '{title}' (Category: {category}) ===")
    run = simulate_ticket_routing(category, title)
    run.print_history()

## 6. Pattern 3 — Apply to Each with Error Handling

Simulates batch email processing where some items may fail.

**Power Automate equivalent:**
- Get items from SharePoint
- Apply to Each: Send email to each person
- Some recipients have invalid email addresses (simulated failure)

In [ ]:
# Sample SharePoint list items — some have invalid emails to simulate failures
TEAM_MEMBERS = [
    {"name": "Alice Chen", "email": "alice@contoso.com", "department": "Finance"},
    {"name": "Bob Rivera", "email": "INVALID_EMAIL", "department": "HR"},
    {"name": "Carol Kim", "email": "carol@contoso.com", "department": "Finance"},
    {"name": "Dan Okafor", "email": "dan@contoso.com", "department": "IT"},
    {"name": "Eva Santos", "email": "", "department": "Finance"},  # Empty email
]


def send_reminder_email(member: dict) -> dict:
    """
    Simulates the 'Send an email' action inside Apply to Each.
    Raises an exception for invalid or empty email addresses.
    """
    email = member["email"]
    if not email:
        raise ValueError(f"Email address is empty for {member['name']}")
    if "@" not in email:
        raise ValueError(f"Invalid email address '{email}' for {member['name']}")
    # Valid email — simulate success
    return {"sent_to": email, "subject": "Monthly reminder"}


def simulate_batch_email() -> FlowRun:
    """Simulate sending reminders to all team members."""
    run = FlowRun(
        trigger_name="Scheduled: Monthly reminder",
        trigger_data={"frequency": "monthly"},
    )
    executor = FlowExecutor(run)

    # Get items from SharePoint
    members_ref = []  # Will be populated by the action
    executor.run_action(
        "Get items from SharePoint",
        fn=lambda: TEAM_MEMBERS,
    )

    # Apply to Each — sends email to each member
    def loop_body(member: dict, index: int) -> None:
        result = send_reminder_email(member)
        # Record individual iteration success
        action_result = ActionResult(
            name=f"Send email to {member['name']}",
            status=ActionStatus.SUCCEEDED,
            output=result,
        )
        run.record(action_result)

    executor.apply_to_each(
        name="Apply to each team member",
        items=TEAM_MEMBERS,
        body=loop_body,
    )

    return run


print("=== Batch Email Flow Run ===")
run = simulate_batch_email()
run.print_history()

## 7. Pattern 4 — Do Until (Polling)

Simulates polling an external system until a job completes.

**Power Automate equivalent:**
- HTTP action inside Do Until: GET /api/jobs/123/status
- Exit condition: `Job_Status` is equal to `Completed`
- Count limit: 10 iterations
- Inside loop: Delay action (30 seconds in real flows; simulated here)

In [ ]:
def simulate_job_polling(succeed_on_iteration: int = 4) -> FlowRun:
    """
    Simulate polling a job status API until it reports 'Completed'.

    Parameters
    ----------
    succeed_on_iteration : int
        The poll number on which the job will report Completed.
        Use a value > max_count to simulate a job that never completes.
    """
    run = FlowRun(
        trigger_name="HTTP request: Start batch job",
        trigger_data={"job_id": "batch-7741"},
    )
    executor = FlowExecutor(run)

    # State shared between Do Until body and condition
    job_status = {"value": "Running", "poll_count": 0}

    # Trigger the job
    executor.run_action(
        "Start batch job",
        fn=lambda: {"job_id": "batch-7741", "status": "Running"},
    )

    # Do Until: poll until Completed or limit reached
    def poll_body(iteration: int) -> None:
        job_status["poll_count"] = iteration
        # Simulate the job completing on the expected iteration
        if iteration >= succeed_on_iteration:
            job_status["value"] = "Completed"
        status_record = ActionResult(
            name=f"HTTP GET /jobs/batch-7741/status (poll {iteration})",
            status=ActionStatus.SUCCEEDED,
            output={"status": job_status["value"], "progress": min(iteration * 25, 100)},
        )
        run.record(status_record)

    executor.do_until(
        name="Poll until Completed",
        condition_fn=lambda: job_status["value"] == "Completed",
        body=poll_body,
        max_count=10,  # Safety limit
    )

    # Action after the loop
    executor.run_action(
        "Download job results",
        fn=lambda: {"rows_processed": 1547, "errors": 0},
    )

    return run


print("=== Job Polling: succeeds on iteration 4 ===")
run = simulate_job_polling(succeed_on_iteration=4)
run.print_history()

print()
print("=== Job Polling: never completes (hits count limit) ===")
run_timeout = simulate_job_polling(succeed_on_iteration=999)
run_timeout.print_history()

## 8. Pattern 5 — Scope: Try / Catch / Finally

Simulates structured error handling using three Scope actions.

**Power Automate equivalent:**
- Try Scope: main order processing actions
- Catch Scope (Run After: Failed, Timed Out): log error, alert operations
- Finally Scope (Run After: Succeeded + Failed + Timed Out): write audit log

In [ ]:
def simulate_order_processing(order_id: str, force_failure: bool = False) -> FlowRun:
    """
    Simulate order processing with Try/Catch/Finally error handling.

    Parameters
    ----------
    order_id : str
        The order identifier.
    force_failure : bool
        If True, the inventory check will raise an exception to trigger the Catch scope.
    """
    run = FlowRun(
        trigger_name="New order created",
        trigger_data={"order_id": order_id},
    )
    executor = FlowExecutor(run)

    # Track whether the Try scope succeeded for the Finally scope
    try_scope_status = {"succeeded": True, "error": None}

    # --- TRY SCOPE ---
    try:
        def try_body() -> None:
            executor.run_action(
                "Get order details",
                fn=lambda: {"order_id": order_id, "items": 3, "total": 450.00},
            )
            executor.run_action(
                "Check inventory",
                fn=lambda: (_ for _ in ()).throw(
                    RuntimeError(f"Item SKU-9182 out of stock for order {order_id}")
                ) if force_failure else {"in_stock": True, "reserved": True},
            )
            executor.run_action(
                "Create shipping label",
                fn=lambda: {"label_id": "LBL-44892", "carrier": "FedEx"},
            )

        try_result = executor.scope("Try", try_body)

        if try_result.status == ActionStatus.FAILED:
            try_scope_status["succeeded"] = False
            try_scope_status["error"] = try_result.error
            raise RuntimeError(try_result.error or "Try scope failed")

    except RuntimeError:
        # --- CATCH SCOPE ---
        # Configured Run After: Failed + Timed Out (not Succeeded)
        catch_error = try_scope_status["error"]
        executor.run_action(
            "Catch — Log error details",
            fn=lambda: {"logged_error": catch_error, "order_id": order_id},
        )
        executor.run_action(
            "Catch — Send alert to operations",
            fn=lambda: {
                "email_sent_to": "ops@contoso.com",
                "subject": f"Order {order_id} processing failed",
                "body": catch_error,
            },
        )
        executor.run_action(
            "Catch — Update order status to Error",
            fn=lambda: {"order_id": order_id, "status": "Error"},
        )

    finally:
        # --- FINALLY SCOPE ---
        # Configured Run After: Succeeded + Failed + Timed Out
        executor.run_action(
            "Finally — Write audit log",
            fn=lambda: {
                "order_id": order_id,
                "outcome": "Success" if try_scope_status["succeeded"] else "Error",
                "timestamp": "2024-03-08T09:15:00Z",
            },
            # Finally always runs — override run_after to include all states
            run_after=[ActionStatus.SUCCEEDED, ActionStatus.FAILED, ActionStatus.TIMED_OUT],
        )

    return run


print("=== Order Processing: success path ===")
run_ok = simulate_order_processing("ORD-2024-0055", force_failure=False)
run_ok.print_history()

print()
print("=== Order Processing: failure path (out of stock) ===")
run_fail = simulate_order_processing("ORD-2024-0056", force_failure=True)
run_fail.print_history()

## 9. Visualizing Flow Execution Paths

This section renders the run history as a status timeline — similar to what Power Automate shows in the run detail view.

In [ ]:
# Color palette mirrors Power Automate's status colors
STATUS_COLORS = {
    ActionStatus.SUCCEEDED: "#107C10",   # Green
    ActionStatus.FAILED: "#D83B01",       # Red-orange
    ActionStatus.SKIPPED: "#8A8886",      # Grey
    ActionStatus.TIMED_OUT: "#CA5010",   # Amber-orange
}


def plot_run_history(run: FlowRun, title: str = "") -> None:
    """
    Render a flow run's action history as a horizontal timeline.

    Each action is a colored bar indicating its status. This mirrors
    the timeline view in Power Automate's run detail panel.
    """
    results = run.action_results
    if not results:
        print("No actions recorded.")
        return

    n = len(results)
    fig, ax = plt.subplots(figsize=(12, max(3, n * 0.55)))

    for i, result in enumerate(results):
        y = n - i - 1  # Draw top-to-bottom
        color = STATUS_COLORS[result.status]

        # Action bar
        ax.barh(y, 1, left=0, height=0.6, color=color, alpha=0.85, edgecolor="white", linewidth=1.5)

        # Action name (left of bar)
        ax.text(
            -0.02, y, result.name,
            ha="right", va="center",
            fontsize=9, color="#323130",
        )

        # Status label (inside bar)
        ax.text(
            0.5, y, result.status.value,
            ha="center", va="center",
            fontsize=8.5, color="white", fontweight="bold",
        )

        # Error message (right of bar, if any)
        if result.error:
            error_short = result.error[:60] + "..." if len(result.error) > 60 else result.error
            ax.text(
                1.02, y, error_short,
                ha="left", va="center",
                fontsize=7.5, color="#D83B01", style="italic",
            )

    # Legend
    legend_handles = [
        mpatches.Patch(color=color, label=status.value)
        for status, color in STATUS_COLORS.items()
    ]
    ax.legend(handles=legend_handles, loc="lower right", fontsize=8, framealpha=0.9)

    ax.set_xlim(-1.5, 3.5)
    ax.set_ylim(-0.8, n - 0.2)
    ax.axis("off")
    ax.set_title(
        title or f"Run history: {run.trigger_name}",
        fontsize=11, fontweight="bold", pad=12, color="#201f1e"
    )

    plt.tight_layout()
    plt.show()


# Visualize the success and failure paths for the order processing flow
plot_run_history(run_ok, "Order Processing — Success Path")
plot_run_history(run_fail, "Order Processing — Failure Path (Catch + Finally triggered)")

## 10. Comparing Control Flow Patterns Side by Side

This chart shows all four flow patterns across a set of scenarios. Each column is a simulated run; each row is an action.

In [ ]:
# Run all patterns and collect for comparison
runs_to_compare = [
    (simulate_invoice_routing(3200), "Invoice\n$3,200 (auto-approve)"),
    (simulate_invoice_routing(12500), "Invoice\n$12,500 (CFO approval)"),
    (simulate_ticket_routing("Security", "Breach detected"), "Ticket\nSecurity"),
    (simulate_ticket_routing("Printing", "Printer issue"), "Ticket\nDefault"),
]

fig, axes = plt.subplots(1, len(runs_to_compare), figsize=(16, 5), sharey=False)

for ax, (run, label) in zip(axes, runs_to_compare):
    results = run.action_results
    n = len(results)

    for i, result in enumerate(results):
        y = n - i - 1
        color = STATUS_COLORS[result.status]
        ax.barh(y, 0.8, left=0.1, height=0.6, color=color, alpha=0.85, edgecolor="white", linewidth=1)
        short_name = result.name[:22] + "..." if len(result.name) > 22 else result.name
        ax.text(0.5, y, short_name, ha="center", va="center", fontsize=6.5, color="white", fontweight="bold")

    ax.set_xlim(0, 1)
    ax.set_ylim(-0.6, n - 0.4)
    ax.axis("off")
    ax.set_title(label, fontsize=8.5, fontweight="bold", color="#201f1e", pad=6)

# Shared legend
legend_handles = [
    mpatches.Patch(color=color, label=status.value)
    for status, color in STATUS_COLORS.items()
]
fig.legend(handles=legend_handles, loc="lower center", ncol=4, fontsize=8, framealpha=0.9)
fig.suptitle("Power Automate Control Flow Patterns — Simulated Run Histories", fontsize=11, fontweight="bold")
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

## 11. Exercise: Implement a Flow Pattern

**Scenario:** A scheduled flow runs every weekday morning and processes expense reports submitted in the last 24 hours.

For each expense report:
- If the amount is **under $100**: auto-approve
- If the amount is **$100 to $999**: route to the submitter's direct manager
- If the amount is **$1,000 or more**: route to Finance and flag for CFO review

If any approval routing action fails, log the failure and continue to the next report (don't stop the whole loop).

Implement this using `apply_to_each` with `condition` calls inside the loop body.

In [ ]:
EXPENSE_REPORTS = [
    {"id": "EXP-001", "submitter": "Alice Chen", "amount": 45.00, "manager": "Bob Rivera"},
    {"id": "EXP-002", "submitter": "Carol Kim", "amount": 350.00, "manager": "Dan Okafor"},
    {"id": "EXP-003", "submitter": "Dan Okafor", "amount": 1250.00, "manager": "Eva Santos"},
    {"id": "EXP-004", "submitter": "Eva Santos", "amount": 78.00, "manager": "Alice Chen"},
    {"id": "EXP-005", "submitter": "Frank Liu", "amount": 2100.00, "manager": "Alice Chen"},
]


def simulate_expense_routing() -> FlowRun:
    """
    YOUR IMPLEMENTATION HERE.

    Use executor.apply_to_each() to loop over EXPENSE_REPORTS.
    Use executor.condition() inside the loop body to route by amount.

    Routing rules:
    - amount < 100: auto-approve
    - 100 <= amount < 1000: route to manager
    - amount >= 1000: route to Finance + flag for CFO

    Returns
    -------
    FlowRun
        Completed flow run with full action history.
    """
    run = FlowRun(
        trigger_name="Scheduled: Daily expense processing",
        trigger_data={"reports_count": len(EXPENSE_REPORTS)},
    )
    executor = FlowExecutor(run)

    # Step 1: Get expense reports (already loaded in EXPENSE_REPORTS)
    executor.run_action(
        "Get expense reports from SharePoint",
        fn=lambda: {"count": len(EXPENSE_REPORTS)},
    )

    # Step 2: YOUR CODE — apply_to_each over EXPENSE_REPORTS
    # Inside the loop body, use nested conditions to implement the routing rules
    # Hint: Use two conditions:
    #   First condition: amount < 100 → auto-approve / else → check further
    #   Second condition (in the No branch): amount < 1000 → manager / else → Finance + CFO

    # ... your implementation ...

    executor.run_action(
        "Send daily processing summary",
        fn=lambda: {"reports_processed": len(EXPENSE_REPORTS), "summary_sent": True},
    )

    return run


# Uncomment to run your implementation:
# run = simulate_expense_routing()
# run.print_history()
# plot_run_history(run, "Expense Report Routing — Daily Processing")

print("Implement simulate_expense_routing() above, then uncomment the last three lines to test.")

In [ ]:
# Reference implementation — review only after attempting the exercise

def simulate_expense_routing_solution() -> FlowRun:
    """Reference implementation of the expense routing flow."""
    run = FlowRun(
        trigger_name="Scheduled: Daily expense processing",
        trigger_data={"reports_count": len(EXPENSE_REPORTS)},
    )
    executor = FlowExecutor(run)

    executor.run_action(
        "Get expense reports from SharePoint",
        fn=lambda: {"count": len(EXPENSE_REPORTS)},
    )

    def process_report(report: dict, index: int) -> None:
        amount = report["amount"]
        report_id = report["id"]
        submitter = report["submitter"]
        manager = report["manager"]

        # First condition: under $100?
        executor.condition(
            name=f"{report_id} — Amount < $100?",
            expression=amount < 100,
            yes_branch=lambda: executor.run_action(
                f"{report_id} — Auto-approve (${amount:.2f})",
                fn=lambda: {"status": "Approved", "method": "Auto", "amount": amount},
            ),
            # No branch: check the higher tier
            no_branch=lambda: executor.condition(
                name=f"{report_id} — Amount < $1,000?",
                expression=amount < 1000,
                yes_branch=lambda: executor.run_action(
                    f"{report_id} — Route to manager {manager}",
                    fn=lambda: {"routed_to": manager, "amount": amount, "submitter": submitter},
                ),
                no_branch=lambda: [
                    executor.run_action(
                        f"{report_id} — Route to Finance (${amount:.2f})",
                        fn=lambda: {"routed_to": "finance@contoso.com", "amount": amount},
                    ),
                    executor.run_action(
                        f"{report_id} — Flag for CFO review",
                        fn=lambda: {"flagged": True, "cfo_notified": True, "amount": amount},
                    ),
                ],
            ),
        )

    executor.apply_to_each(
        name="Apply to each expense report",
        items=EXPENSE_REPORTS,
        body=process_report,
    )

    executor.run_action(
        "Send daily processing summary",
        fn=lambda: {"reports_processed": len(EXPENSE_REPORTS), "summary_sent": True},
    )

    return run


print("=== Expense Routing — Reference Solution ===")
run_solution = simulate_expense_routing_solution()
run_solution.print_history()
plot_run_history(run_solution, "Expense Report Routing — Daily Processing (Reference Solution)")

In [ ]:
section_divider("Power Automate vs. Python: Pattern Mapping")

## 12. Power Automate vs. Python: Pattern Mapping

Every Power Automate control flow pattern has a direct Python equivalent. Understanding this mapping helps you think clearly about what a flow is doing.

| Power Automate | Python | Key difference |
|---|---|---|
| **Condition** | `if/else` | Same logic; PA adds run history tracking |
| **Switch** | `match/case` or `dict.get()` | PA Switch only supports equality |
| **Apply to Each** | `for item in items` | PA tracks each iteration in run history |
| **Do Until** | `while not condition` | PA adds Count and Timeout limits |
| **Scope (Try)** | `try:` block | PA groups for collective error status |
| **Scope (Catch)** | `except:` block | PA uses Run After instead of exception type |
| **Scope (Finally)** | `finally:` block | Same semantics |
| **Configure Run After** | `except`/`finally` clause | PA applies to any action, not just code blocks |
| **Terminate** | `sys.exit()` or `raise SystemExit` | PA sets a final status (Succeeded/Failed/Cancelled) |
| **Retry Policy** | `@retry` decorator | PA applies at the action level, not function level |

---

## Summary

**What you built:**

1. A `FlowExecutor` that simulates Power Automate's core control flow primitives
2. Simulations of four real-world flow patterns: invoice routing, ticket routing, batch email, job polling, and order processing with error handling
3. Visualizations of run history timelines — the same information Power Automate shows in its run detail view

**Key takeaways:**

- Every action in Power Automate ends in one of four states: Succeeded, Failed, Skipped, Timed Out
- Conditions route to Yes/No; Switch routes to named cases by equality; Apply to Each iterates; Do Until polls
- Configure Run After determines which state combinations trigger each subsequent action
- Scope + Run After implements try/catch/finally with any granularity

**Next:** Module 05 — SharePoint and Excel flows will apply all of these patterns to real connector actions.

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])